# LSTM Autoencoder — NDVI-Only Model (Production)

**Purpose:** Train the production LSTM Autoencoder using *only* Sentinel-2 NDVI time-series.  
**Justification:** Feature ablation in `lstm_anomaly_detection.ipynb` showed that adding weather features **degraded AUC by 8.8%** (0.619 vs 0.708). The NDVI-only model is simpler, faster at inference, and requires no external weather API calls.  
**Threshold fix:** The original model used the 95th-percentile threshold, producing 276 false positives (Precision=53%, Recall=95%). This notebook uses the **99th percentile of reconstruction MSE on normal training windows** to achieve a much stricter decision boundary.  

---
## Architecture
```
Input:   (batch, 30, 1)    — 30 NDVI observations, 1 feature
Encoder: LSTM(1, 64, layers=2) → Linear(64, 16)  [latent]
Decoder: Repeat(30) → LSTM(16, 64, layers=2) → Linear(64, 1)
Output:  (batch, 30, 1)    — reconstructed NDVI sequence
Loss:    MSE(input, reconstruction)
Anomaly: MSE > threshold  (99th pct of normal-window training MSE)
```


In [ ]:
import json
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, roc_auc_score
from torch.utils.data import DataLoader, TensorDataset

NOTEBOOK_DIR = Path(".").resolve()
BACKEND_DIR  = NOTEBOOK_DIR.parent
sys.path.insert(0, str(BACKEND_DIR))

from app.ml.lstm_autoencoder import AnomalyDetector, FeatureScaler, LSTMAutoencoder

# ── Hyperparameters ─────────────────────────────────────────────────────
WINDOW_SIZE = 30
INPUT_SIZE  = 1
HIDDEN_SIZE = 64
NUM_LAYERS  = 2
LATENT_DIM  = 16
DROPOUT     = 0.1
BATCH_SIZE  = 32
LR          = 1e-3
EPOCHS      = 200
PATIENCE    = 20
SEED        = 42

DATA_PATH  = BACKEND_DIR / "data" / "training" / "ndvi_gpx_real.parquet"
MODEL_DIR  = BACKEND_DIR / "data" / "models"

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"PyTorch {torch.__version__}  |  NumPy {np.__version__}  |  Pandas {pd.__version__}")


## 1  Data Loading


In [ ]:
df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["parcel_id", "date"]).reset_index(drop=True)
print(f"Shape: {df.shape}  |  Parcels: {df['parcel_id'].nunique()}")
df.head()


## 2  Anomaly Labeling (for evaluation only)

A record is anomalous if its NDVI is >2.5 std below its parcel's rolling mean **OR**
below the month-level seasonal expectation. The model is **trained only on normal windows**.


In [ ]:
SEASONAL_NDVI = {
    1: 0.27, 2: 0.31, 3: 0.48, 4: 0.64, 5: 0.71, 6: 0.74,
    7: 0.67, 8: 0.54, 9: 0.44, 10: 0.38, 11: 0.32, 12: 0.26,
}
SEASONAL_STD = 0.14


def compute_anomaly_flags(subdf):
    ndvi  = subdf["ndvi_mean"].values.astype(float)
    months = subdf["date"].dt.month.values
    s = pd.Series(ndvi)
    roll_mean = s.rolling(10, min_periods=3, center=True).mean().to_numpy()
    roll_std  = (s.rolling(10, min_periods=3, center=True).std()
                  .fillna(0.05).clip(lower=0.02).to_numpy())
    seasonal = np.array([SEASONAL_NDVI[m] for m in months], dtype=float)
    z_roll = (ndvi - roll_mean) / roll_std
    z_seas = (ndvi - seasonal) / SEASONAL_STD
    return (z_roll < -2.5) | (z_seas < -2.5)


flags = np.zeros(len(df), dtype=bool)
for pid, idx in df.groupby("parcel_id").groups.items():
    flags[idx] = compute_anomaly_flags(df.loc[idx])
df["is_anomaly"] = flags
print(f"Anomaly rate: {df['is_anomaly'].mean():.1%}  ({df['is_anomaly'].sum()} / {len(df)})")


## 3  Parcel-Level Split (14 / 3 / 3)


In [ ]:
parcels  = sorted(df["parcel_id"].unique())
rng      = np.random.default_rng(SEED)
shuffled = rng.permutation(parcels)
N_TRAIN, N_VAL, N_TEST = 14, 3, 3
train_parcels = set(shuffled[:N_TRAIN])
val_parcels   = set(shuffled[N_TRAIN:N_TRAIN + N_VAL])
test_parcels  = set(shuffled[N_TRAIN + N_VAL:])
print("Train:", sorted(train_parcels))
print("Val:  ", sorted(val_parcels))
print("Test: ", sorted(test_parcels))


## 4  Sliding-Window Construction

Window size = 30 NDVI observations (~150 days at 5-day revisit).  
Label = anomaly of the **last observation** in the window.


In [ ]:
def build_windows(df_subset, window_size=WINDOW_SIZE):
    windows, labels = [], []
    for _pid, group in df_subset.groupby("parcel_id"):
        group   = group.sort_values("date")
        ndvi    = group["ndvi_mean"].values.astype(np.float32)
        anomaly = group["is_anomaly"].values
        if len(ndvi) < window_size:
            continue
        for i in range(len(ndvi) - window_size + 1):
            windows.append(ndvi[i : i + window_size])
            labels.append(int(anomaly[i + window_size - 1]))
    return np.array(windows, np.float32), np.array(labels, np.int32)


X_train, y_train = build_windows(df[df["parcel_id"].isin(train_parcels)])
X_val,   y_val   = build_windows(df[df["parcel_id"].isin(val_parcels)])
X_test,  y_test  = build_windows(df[df["parcel_id"].isin(test_parcels)])

for name, X, y in [("Train", X_train, y_train), ("Val", X_val, y_val), ("Test", X_test, y_test)]:
    print(f"{name:5s}: {X.shape}  anomaly={y.mean():.1%}")


## 5  MinMax Normalisation (fit on normal train windows)


In [ ]:
train_normal_mask = y_train == 0
ndvi_min = float(X_train[train_normal_mask].min())
ndvi_max = float(X_train[train_normal_mask].max())

def normalize(X):
    return (X - ndvi_min) / max(ndvi_max - ndvi_min, 1e-6)

X_train_n = normalize(X_train)
X_val_n   = normalize(X_val)
X_test_n  = normalize(X_test)
print(f"NDVI range: [{ndvi_min:.4f}, {ndvi_max:.4f}]")


## 6  Dataset & DataLoaders


In [ ]:
def to_tensor(X):
    return torch.FloatTensor(X).unsqueeze(-1)  # (N,W) -> (N,W,1)

T_train_all    = to_tensor(X_train_n)
T_train_normal = T_train_all[train_normal_mask]
T_val          = to_tensor(X_val_n)
T_test         = to_tensor(X_test_n)

train_loader = DataLoader(TensorDataset(T_train_normal), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(T_val),          batch_size=BATCH_SIZE, shuffle=False)

print(f"Training on {len(T_train_normal)} normal windows "
      f"(skipped {(~train_normal_mask).sum()} anomaly windows)")


## 7  Model


In [ ]:
model = LSTMAutoencoder(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    latent_dim=LATENT_DIM,
    window_size=WINDOW_SIZE,
    dropout=DROPOUT,
)
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")


## 8  Training


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=8, factor=0.5)
criterion = nn.MSELoss()

best_val_loss, best_state, patience_counter = float("inf"), None, 0
train_hist, val_hist = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    batch_losses = []
    for (batch,) in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(batch), batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        batch_losses.append(loss.item())
    model.eval()
    val_losses = []
    with torch.no_grad():
        for (batch,) in val_loader:
            val_losses.append(criterion(model(batch), batch).item())
    tl, vl = np.mean(batch_losses), np.mean(val_losses)
    train_hist.append(tl); val_hist.append(vl)
    scheduler.step(vl)
    if vl < best_val_loss:
        best_val_loss = vl
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}"); break
    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | train={tl:.6f} | val={vl:.6f} | best={best_val_loss:.6f}")

model.load_state_dict(best_state)
model.eval()
print(f"\nBest val loss: {best_val_loss:.8f}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_hist, label="Train", color="steelblue")
ax.plot(val_hist,   label="Val",   color="tomato")
ax.set_xlabel("Epoch"); ax.set_ylabel("MSE")
ax.set_title("NDVI-Only LSTM — Training Curves")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(BACKEND_DIR / "data" / "training" / "ndvi_only_training_curves.png", dpi=120)
plt.show()


## 9  Anomaly Threshold

Selected as `max(p99, mean + 3*std)` on normal training-window reconstruction MSE.  
This is considerably stricter than the old 95th-percentile threshold that caused 276 FP.


In [ ]:
with torch.no_grad():
    recon_normal = model(T_train_normal)
    mse_normal   = ((T_train_normal - recon_normal) ** 2).mean(dim=(1, 2)).numpy()

p99       = float(np.percentile(mse_normal, 99))
mean3std  = float(mse_normal.mean() + 3.0 * mse_normal.std())
threshold = max(p99, mean3std)

print(f"MSE on normal windows: mean={mse_normal.mean():.6f}  std={mse_normal.std():.6f}")
print(f"  p95 (old):  {np.percentile(mse_normal, 95):.6f}")
print(f"  p99:        {p99:.6f}")
print(f"  mean+3std:  {mean3std:.6f}")
print(f"  SELECTED:   {threshold:.6f}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(mse_normal, bins=30, color="steelblue", alpha=0.7, label="Normal window MSE")
ax.axvline(threshold,                       color="red",    ls="--", label=f"Threshold p99={threshold:.5f}")
ax.axvline(np.percentile(mse_normal, 95),   color="orange", ls=":",  label="Old threshold (p95)")
ax.set_xlabel("Reconstruction MSE"); ax.set_ylabel("Count")
ax.set_title("MSE Distribution — Normal Training Windows")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(BACKEND_DIR / "data" / "training" / "ndvi_only_threshold.png", dpi=120)
plt.show()


## 10  Test Set Evaluation


In [ ]:
with torch.no_grad():
    mse_test  = ((T_test  - model(T_test))  ** 2).mean(dim=(1, 2)).numpy()
    mse_train = ((T_train_all - model(T_train_all)) ** 2).mean(dim=(1, 2)).numpy()

y_pred = (mse_test > threshold).astype(int)
print(classification_report(y_test, y_pred, target_names=["Normal", "Anomaly"]))
if len(np.unique(y_test)) > 1:
    auc = float(roc_auc_score(y_test, mse_test))
    print(f"ROC AUC: {auc:.4f}")
else:
    auc = 0.0
    print("ROC AUC: N/A")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, mse, y, title in [
    (axes[0], mse_train, y_train, "Train"),
    (axes[1], mse_test,  y_test,  "Test"),
]:
    ax.hist(mse[y == 0], bins=25, alpha=0.6, color="steelblue", label="Normal")
    ax.hist(mse[y == 1], bins=25, alpha=0.6, color="tomato",    label="Anomaly")
    ax.axvline(threshold, color="black", ls="--", label="Threshold")
    ax.set_title(f"{title}: MSE Distribution"); ax.set_xlabel("MSE")
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(BACKEND_DIR / "data" / "training" / "ndvi_only_mse_dist.png", dpi=120)
plt.show()


## 11  Save Artifacts


In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

torch.save(model.state_dict(), MODEL_DIR / "lstm_autoencoder.pt")

scaler = FeatureScaler(
    feature_names=["ndvi_mean"],
    min_vals=np.array([ndvi_min], dtype=np.float32),
    max_vals=np.array([ndvi_max], dtype=np.float32),
)
scaler.to_json(str(MODEL_DIR / "scaler_params.json"))

tp = int(((y_pred == 1) & (y_test == 1)).sum())
fp = int(((y_pred == 1) & (y_test == 0)).sum())
fn = int(((y_pred == 0) & (y_test == 1)).sum())
tn = int(((y_pred == 0) & (y_test == 0)).sum())
precision = tp / max(tp + fp, 1)
recall    = tp / max(tp + fn, 1)
f1 = 2 * precision * recall / max(precision + recall, 1e-9)

config = {
    "input_size":          INPUT_SIZE,
    "hidden_size":         HIDDEN_SIZE,
    "num_layers":          NUM_LAYERS,
    "latent_dim":          LATENT_DIM,
    "window_size":         WINDOW_SIZE,
    "dropout":             0.0,
    "anomaly_threshold":   threshold,
    "threshold_strategy":  "max(p99, mean+3std) on normal training windows",
    "feature_names":       ["ndvi_mean"],
    "training_date":       datetime.now().isoformat(),
    "training_parcels":    len(parcels),
    "parcel_split":        {"train": N_TRAIN, "val": N_VAL, "test": N_TEST},
    "data_source":         "real_apia_gpx",
    "roc_auc":             round(auc, 6),
    "best_f1":             round(f1, 6),
    "precision":           round(precision, 4),
    "recall":              round(recall, 4),
    "false_positives":     fp,
    "best_val_loss":       round(best_val_loss, 10),
    "mse_normal_p99":      round(p99, 10),
    "mse_normal_mean3std": round(mean3std, 10),
    "epochs_trained":      len(train_hist),
}
with open(MODEL_DIR / "model_config.json", "w") as f:
    json.dump(config, f, indent=2)

for fn_ in ["lstm_autoencoder.pt", "scaler_params.json", "model_config.json"]:
    p = MODEL_DIR / fn_
    print(f"  {fn_:30s}  {p.stat().st_size // 1024} KB")
print(json.dumps({k: v for k, v in config.items() if k not in ("parcel_split",)}, indent=2))


## 12  Backend Inference Smoke Test


In [ ]:
import app.ml.lstm_autoencoder as _ml
_ml._detector_instance = None  # reset singleton

detector = AnomalyDetector.load(str(MODEL_DIR))
print(f"Loaded: input_size={detector.config['input_size']}, threshold={detector.threshold:.8f}")

healthy = [0.6 + 0.05 * np.sin(i * 0.3) for i in range(50)]
s, flag, _ = detector.predict(healthy)
print(f"Healthy ndvi series: score={s:.6f}, is_anomaly={flag}")

# Sudden crop-failure drop in the middle
anomalous = [0.65] * 20 + [0.10] * 10 + [0.65] * 20
s2, flag2, _ = detector.predict(anomalous)
print(f"Drop event  series: score={s2:.6f}, is_anomaly={flag2}")
